In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from processing import *
from wavelengths import *
from fit_prefilter import *
#from fit_voigt import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'

folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2025-09-16/'
#folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2024-07-10/'
files = sorted(glob.glob(folder + '*.fits.gz'))

#for file in files:
#    process(file, dark_file=dark_file,
#            _calc_wavelengths=True, _find_center=True, _mask=True, _rebin=8, to_file=True)

In [3]:
files = sorted(glob.glob('*.fits'))

In [4]:
k_T = 0.03

wvlns = []
temperatures = []
datas = []
mus = []

for file in files:
    with fits.open(file) as hdul:
        data = hdul[0].data
        header = hdul[0].header

    nx, ny = header['NAXIS2'], header['NAXIS1']
    xc, yc = header['CRPIX2'] - 1, header['CRPIX1'] - 1
    rsun = header['RSUN_ARC'] / header['CDELT1']

    xi, yi = np.mgrid[:nx,:ny]
    mu = np.sqrt((rsun ** 2 - (xi - xc) ** 2 - (yi - yc) ** 2).clip(0)) / rsun

    datas += [data]
    wvlns += [read_wavelengths(header)]
    temperatures += [header['FGOV1PT1']]
    mus += [mu]

datas = np.swapaxes(np.concat(datas, axis=1), 0, 1)
wvlns = np.array(wvlns)
temperatures = np.array(temperatures)
mus = np.array(mus)

wvlns -= k_T * (np.expand_dims(temperatures, -1) - 61)

In [5]:
dwv = 0.01

a, b = np.max(np.min(wvlns, axis=-1)), np.min(np.max(wvlns, axis=-1))
a = np.round(a + dwv / 2, 2)
b = np.round(b - dwv / 2, 2)

wv = np.arange(a, b + dwv / 2, dwv)

In [6]:
from fitting import interpolate

Q = []
for i in range(3):
    Q += [interpolate(datas[i], wvlns[i], np.expand_dims(wv, (-1,-2)))]
Q = np.array(Q)

In [7]:
x, y = 50, 55

plt.figure(figsize=(10,8))
for i in range(3):
    plt.plot(wv, Q[i,:,x,y])
plt.tight_layout()

In [8]:
params, gamma, res = fit_prefilter(Q[1,:], Q[2,:], wv,
                   axis=0, lam=1e-6, niter=5)

print(gamma, np.nanmedian(res))

[0.04583075 0.05613994 0.30871052] 71.03093520778992


/home/ulyanov/miniconda3/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [9]:
params.shape

(2, 256, 256)

In [10]:
params1 = np.expand_dims(params, 1)
params2 = params1.copy()
params2[0] += gamma[-1]

Q1 = Q[1] / voigt_func((np.expand_dims(wv, (1,2)), 0), *params1, *gamma)[0]
Q2 = Q[2] / voigt_func((np.expand_dims(wv, (1,2)), 0), *params2, *gamma)[0]

In [11]:
x, y = 60,60

plt.figure(figsize=(10,8))
plt.plot(wv, Q[1,:,x,y])
plt.plot(wv, Q[2,:,x,y])
plt.plot(wv, Q1[:,x,y])
plt.plot(wv, Q2[:,x,y])
plt.tight_layout()

In [12]:
plt.figure(figsize=(10,10))
plt.imshow(params[0], 'seismic', vmin=np.min(wv), vmax=np.max(wv))
plt.tight_layout()

In [13]:
plt.figure(figsize=(10,10))
plt.imshow(-params[1], 'inferno', vmin=0.05, vmax=0.08)
plt.tight_layout()

In [46]:
plt.figure(figsize=(10,10))
plt.imshow(params[2], 'inferno', vmin=0.03, vmax=0.06)
plt.tight_layout()

In [43]:
plt.figure(figsize=(10,10))
plt.imshow(res, 'inferno', vmin=0, vmax=200)
plt.tight_layout()

In [102]:
plt.figure(figsize=(10,10))
plt.imshow(Q[0,-1], 'inferno')
plt.tight_layout()

In [103]:
plt.figure(figsize=(10,10))
plt.imshow(Q[1,-1] / Q[1,0], 'inferno', vmin=0.8, vmax=1.1)
plt.tight_layout()

In [14]:
plt.figure(figsize=(10,10))
plt.imshow(Q2[-1] / Q2[0], 'inferno', vmin=0.8, vmax=1.1)
plt.tight_layout()

In [15]:
plt.figure(figsize=(10,10))
plt.imshow(Q1[-1] / Q1[0], 'inferno', vmin=0.8, vmax=1.1)
plt.tight_layout()